In [1]:
import pandas as pd
import json
import datetime as dt
#from src.utils.formatters import quitar_espacios_str,rellenar_fechas_nulas, rellenar_horarios_nulos, modificar_tipo_fecha

In [ ]:
#df_logistic = pd.read_excel('../data/raw/logistics/logistica_envios.xlsx')
#with open('../data/raw/production/produccion_taller.json', 'r', encoding='utf-8') as archivo:
    # json.load() que sirve para leer archivos en vez de strings
    #data_parsed = json.load(archivo) 

# paso la lista de diccionarios a Pandas para que la aplane
#df_production = pd.json_normalize(data_parsed)






In [3]:
# FUNCIONES
def quitar_espacios_str(df: pd.DataFrame) -> pd.DataFrame:
    # Retornamos el nuevo dataframe modificado
    return df.apply(lambda x: x.str.strip() if x.dtype == 'object' or x.dtype == 'string' else x)
  
def rellenar_fechas_nulas(df: pd.DataFrame, cols: list) -> None:
  for col in cols:
    df[col] = df[col].fillna(pd.to_datetime('1900-01-01'))
    
def rellenar_horarios_nulos(df: pd.DataFrame, cols: list ) -> None:
  for col in cols:
    df[col] = df[col].fillna(pd.to_timedelta('00:00:00'))
    
def modificar_tipo_fecha(df: pd.DataFrame, cols: list) -> None:
  for col in cols:
    df[col] = pd.to_datetime(df[col], format='mixed', dayfirst=True, errors='coerce')
    
def reemplazar_valores(df: pd.DataFrame, col: str ,words_to_replace: list[str], to: str) -> None:
  df[col] = df[col].replace(words_to_replace,to, regex=False)
  
def modificar_tipo_horarios(df: pd.DataFrame, cols: list):
  for col in cols:
    df[col] = pd.to_timedelta(df[col],errors='coerce')

In [ ]:
# 1. Transformación de Sales

df_sales = pd.read_csv('../data/raw/sales/ventas_ecommerce.csv') 

def transform_sales(dataframe: pd.DataFrame) -> pd.DataFrame:
  print(f'Leyendo dataframe {dataframe}')
  
  df = dataframe
  
  # 1. Quita espacios en blanco a todas las columnas str
  df = quitar_espacios_str(df)
  
  # 2. Dividimos la columna fecha_compra en dos columnas para dividir con el tiempo
  df[['fecha_de_compra', 'horario']] = df['fecha_compra'].str.split(' ', expand=True)
  df['fecha_compra'] = pd.to_datetime(df['fecha_compra'], format='mixed', dayfirst=True, errors='coerce')
  df['fecha_de_compra'] = pd.to_datetime(df['fecha_de_compra'], format='mixed', errors='coerce')
  
  # 3. Manejar ingresos negativos
  df.loc[df['ingreso_total'] < 0, 'ingreso_total'] = df['ingreso_total'] * -1
  
  # 4. Transformar horarios en timedelta
  df['horario'] = pd.to_timedelta(df['horario'], errors='coerce')
  
  # 5. Rellenar dechas y horarios nulos
  rellenar_fechas_nulas(df, ['fecha_compra', 'fecha_de_compra'])
  df['horario'] = df['horario'].fillna(pd.to_timedelta('00:00:00'))
  
  return df
  
df_final = transform_sales(df_sales)
df_final.info()
  
  
  
  

Leyendo dataframe       id_pedido id_cliente         fecha_compra          articulo  cantidad  \
0     ORD-00001  CUST-1226  2026-07-20 14:00:00  Remera Sublimada         5   
1     ORD-00002  CUST-1559  2026-07-12 04:00:00  Remera Sublimada         2   
2     ORD-00003   CUST-960  2026-06-14 06:00:00  Remera Sublimada         1   
3     ORD-00004  CUST-1394  2026-07-03 00:00:00      Buzo Canguro         5   
4     ORD-00005  CUST-1230  2026-03-27 06:00:00    Remera Algodón         5   
...         ...        ...                  ...               ...       ...   
5035  ORD-01357   CUST-824           15/05/2026             Gorra         4   
5036  ORD-00533   CUST-674  2026-06-30 18:00:00    Remera Algodón         1   
5037  ORD-03669  CUST-1955  2026-07-05 22:00:00     Taza Cerámica         3   
5038  ORD-03378  CUST-1505  2026-05-25 19:00:00  Remera Sublimada         4   
5039  ORD-01613   CUST-930  2026-05-11 21:00:00  Remera Sublimada         4   

      ingreso_total  
0          

In [37]:
df_logistics = pd.read_excel('../data/raw/logistics/logistica_envios.xlsx')

# Estandarizar provicias
#reemplazar_valores(df_logistics,'provincia_destino',['B. Aires', 'Bs As',],'Buenos Aires')
#reemplazar_valores(df_logistics,'provincia_destino',['Ciudad Autonoma de Buenos Aires'],'CABA')
#reemplazar_valores(df_logistics,'provincia_destino',['Sta Fe'],'Santa Fe')
df_logistics

# Cambiar nombre columna
df_logistics = df_logistics.rename(columns={'fecha_despacho': 'fecha_hora_despacho',
                                            'fecha_entrega': 'fecha_hora_entrega'})


# Dividir columnas de despacho y entrega en fecha y hora
df_logistics[['fecha_despacho', 'hora_despacho']] = df_logistics['fecha_hora_despacho'].str.strip().str.split(' ',expand=True)
df_logistics[['fecha_entrega', 'hora_entrega']] = df_logistics['fecha_hora_entrega'].str.strip().str.split(' ',expand=True)

# Cambiar tipo de datos en fechas y horas
#modificar_tipo_fecha(df_logistics, ['fecha_despacho', 'fecha_entrega'])
#modificar_tipo_horarios(df_logistics, ['hora_despacho', 'hora_entrega'])
#df_logistics.info()

# Asegurarse de valores nulos en costo_envio
df_logistics.loc[df_logistics['costo_envio'] < 0] = df_logistics['costo_envio'] * - 1

# Reemplazar valores nulos
#rellenar_fechas_nulas(df_logistics, ['fecha_despacho', 'fecha_entrega', 'fecha_hora_entrega'])
#rellenar_horarios_nulos(df_logistics, ['hora_despacho', 'hora_entrega'])

df_logistics['orden'] = df_logistics['orden'].str.strip().str.upper()

mascara = df_logistics['orden'].str.startswith('ORD')

df_logistics[~mascara]

df_logistics.loc[~mascara, 'orden'] = "ORD-" + df_logistics['orden']

df_logistics['tracking_id'].str.startswith('TRK').value_counts()



tracking_id
True    4600
Name: count, dtype: int64

In [ ]:
# 2. Transformación de production
with open('../data/raw/production/produccion_taller.json', 'r', encoding='utf-8') as archivo:
    # json.load() que sirve para leer archivos deserializándolo en un objeto de python
    data_parsed = json.load(archivo) 

# paso la lista de diccionarios a Pandas para que la aplane
df_production = pd.json_normalize(data_parsed)

# 1. Cambios de nombres de columnas para más comodidad
df_production = df_production.rename(columns={ 'detalles_taller.maquina_sublimacion': 'id_maquina',
                    'detalles_taller.operador': 'operador',
                    'detalles_taller.inicio_estampado': 'inicio_estampado',
                    'detalles_taller.fin_confeccion': 'fin_confeccion',
                    'detalles_taller.control_calidad_ok': 'control_calidad'})

# 2. Borrar espacios en los str
quitar_espacios_str(df_production)

# 3. Separar la fecha y hora en columnas diferentes
df_production[['fecha_inicio_estampado', 'hora_inicio_estampado']] = df_production['inicio_estampado'].str.split('T', expand=True)

df_production[['fecha_fin_confeccion', 'hora_fin_confeccion']] = df_production['fin_confeccion'].str.split('T', expand=True)

# 4. Pasamos los horarios y las fechas a sus respectivos formatos
modificar_tipo_fecha(df_production,['fecha_inicio_estampado', 'fecha_fin_confeccion'])
modificar_tipo_horarios(df_production,['hora_inicio_estampado', 'hora_fin_confeccion'])


# 5. Completar los nombres nulos en 'Desconocido'
df_production['operador'] = df_production['operador'].fillna('Desconocido')
df_production.info()

# 6. Normalización de referencia de pedido
df_production['pedido_ref'] = df_production['pedido_ref'].str.strip().str.upper()

mascara = df_production['pedido_ref'].str.startswith('ORD')

df_production[~mascara]

df_production.loc[~mascara, 'pedido_ref'] = "ORD-" + df_production['pedido_ref']

# 7. Borrar duplicados
df_production.drop_duplicates()





<class 'pandas.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype          
---  ------                  --------------  -----          
 0   pedido_ref              4600 non-null   str            
 1   id_maquina              4600 non-null   str            
 2   operador                4600 non-null   str            
 3   inicio_estampado        4600 non-null   str            
 4   fin_confeccion          4600 non-null   str            
 5   control_calidad         4600 non-null   bool           
 6   fecha_inicio_estampado  4600 non-null   datetime64[us] 
 7   hora_inicio_estampado   4600 non-null   timedelta64[us]
 8   fecha_fin_confeccion    4600 non-null   datetime64[us] 
 9   hora_fin_confeccion     4600 non-null   timedelta64[us]
dtypes: bool(1), datetime64[us](2), str(5), timedelta64[us](2)
memory usage: 328.1 KB


pedido_ref  id_maquina  operador  inicio_estampado     fin_confeccion       control_calidad  fecha_inicio_estampado  hora_inicio_estampado  fecha_fin_confeccion  hora_fin_confeccion
ORD-00913   MQ-1        Carlos    2026-05-22T13:00:00  2026-05-22T22:00:00  True             2026-05-22              0 days 13:00:00        2026-05-22            0 days 22:00:00        1
ORD-00205   MQ-4        Ana       2026-04-06T00:00:00  2026-04-04T00:00:00  True             2026-06-04              0 days 00:00:00        2026-04-04            0 days 00:00:00        1
ORD-02254   MQ-4        Ana       2026-07-01T09:00:00  2026-07-02T03:00:00  True             2026-01-07              0 days 09:00:00        2026-02-07            0 days 03:00:00        1
ORD-02007   MQ-1        Carlos    2026-04-13T16:00:00  2026-04-14T05:00:00  True             2026-04-13              0 days 16:00:00        2026-04-14            0 days 05:00:00        1
ORD-01829   MQ-4        Carlos    2026-06-21T19:00:00  2026-06-22T21:0